<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.4-a2a/practice/GCP_Capstone_8.4_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 8.4 — A2A Protocol

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: ADK + A2A on Vertex

Run this once. It installs the A2A extras for ADK, authenticates with Application Default Credentials, and points ADK at Vertex AI. Every exercise below builds on this cell.

In [ ]:
!pip install -q 'google-adk[a2a]'
import os
from google.colab import auth
auth.authenticate_user()

# Application Default Credentials + Vertex (no API keys)
os.environ['GOOGLE_CLOUD_PROJECT'] = 'documind-ai-YOUR-ID'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'  # global endpoint for Gemini 3.x generation
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'

# INR display rate used later in the currency example
USD_INR = 85
print('ADK + A2A ready (Vertex + ADC)')

## Base agent: DocuMind

The A2A exercises wrap and expose this agent, so define it once here. Two tools, `gemini-3.6-flash`, ADK `LlmAgent`.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext

def search_documents(query: str, tool_context: ToolContext) -> dict:
    """Search documents.
    Args:
        query: Search query.
    """
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report'}]}

def summarize_document(document_id: str, summary_type: str) -> dict:
    """Summarize a document.
    Args:
        document_id: Document ID.
        summary_type: brief/detailed/executive.
    """
    return {'summary': f'Summary of {document_id}'}

root_agent = LlmAgent(
    name='documind',
    model='gemini-3.6-flash',
    description='Document analysis, summarization, and Q&A agent',
    instruction='You are DocuMind AI. Search and summarize documents.',
    tools=[search_documents, summarize_document],
)
print(f'Agent: {root_agent.name}')

## Exercise 1: Write Agent Card

**Difficulty:** Easy

Create `agent-card.json` for DocuMind with 2 skills.

1. Build a dict with `name`, `description`, `url`, `version`, input/output modes, and `capabilities`.
2. Give it a `skills` list with 2 skills — summarization and Q&A — each with `id`, `name`, `description`, `tags`, and `examples`.
3. Write it to `agent-card.json` and print the JSON.

**Expected behaviour:** Valid JSON with name, skills, capabilities.

In [ ]:
import json

agent_card = {
    'name': 'DocuMind Intelligence Agent',
    'description': 'Document analysis, summarization, and Q&A',
    'url': 'https://documind-a2a.run.app/',
    'version': '1.0.0',
    'defaultInputModes': ['text/plain', 'application/pdf'],
    'defaultOutputModes': ['text/plain', 'application/json'],
    'capabilities': {'streaming': True, 'pushNotifications': True},
    'skills': [
        {
            'id': 'document-summarization',
            'name': 'Document Summarization',
            'description': 'Summarize uploaded documents with configurable detail',
            'tags': ['summarization', 'documents'],
            'examples': ['Summarize this quarterly report']
        },
        {
            'id': 'document-qa',
            'name': 'Document Q&A',
            'description': 'Answer questions about documents using RAG',
            'tags': ['qa', 'rag', 'documents'],
            'examples': ['What revenue was reported in Q3?']
        }
    ]
}

with open('agent-card.json', 'w') as f:
    json.dump(agent_card, f, indent=2)
print(json.dumps(agent_card, indent=2))

## Exercise 2: to_a2a() Server

**Difficulty:** Easy

Wrap the agent with `to_a2a()`. Fetch the Agent Card via curl.

1. Import `to_a2a` and convert `root_agent` into an A2A ASGI app in one line.
2. Note the uvicorn command that serves it and the well-known Agent Card URL.
3. Once served, fetch the auto-generated Agent Card with curl.

**Expected behaviour:** Agent Card JSON returned.

In [ ]:
from google.adk.a2a.utils.agent_to_a2a import to_a2a

# ONE LINE: convert ADK agent to A2A server
a2a_app = to_a2a(root_agent, port=8001)
print('A2A server created')
print('Run: uvicorn agent:a2a_app --host 0.0.0.0 --port 8001')
print('Agent Card: http://localhost:8001/.well-known/agent.json')

In [ ]:
%%bash
# Once the server is running (uvicorn agent:a2a_app --port 8001),
# ADK auto-generates the Agent Card from the agent's tools + metadata.
curl -s http://localhost:8001/.well-known/agent.json | head -c 800

## Exercise 3: Send Task via curl

**Difficulty:** Easy

POST `message/send` to the A2A server. Parse the JSON-RPC response.

1. Build a JSON-RPC 2.0 request with `method: message/send` and a user `message` carrying a text part.
2. POST it to the running A2A server with curl.
3. Parse the JSON-RPC response — a Task with `status` and `artifacts`.

**Expected behaviour:** Task response with status + artifacts.

In [ ]:
import json

# A2A message/send request
request = {
    'jsonrpc': '2.0',
    'id': 1,
    'method': 'message/send',
    'params': {
        'message': {
            'role': 'user',
            'parts': [{
                'kind': 'text',
                'text': 'Summarize this contract and flag compliance risks'
            }]
        }
    }
}
print('A2A JSON-RPC request:')
print(json.dumps(request, indent=2))

with open('a2a_request.json', 'w') as f:
    json.dump(request, f)
print('\nSaved to a2a_request.json for the curl POST below.')

**Run these after starting the A2A server** (an earlier cell must launch `a2a_app` on port 8001 in a separate process/terminal). In Colab the server blocks the kernel, so run it in a terminal:

```bash
# POST the task to the running A2A server and pretty-print the JSON-RPC Task result.
curl -s -X POST http://localhost:8001/ \
  -H 'Content-Type: application/json' \
  -d @a2a_request.json | python3 -m json.tool
```

## Exercise 4: RemoteA2aAgent

**Difficulty:** Medium

Connect to a remote A2A agent as a `sub_agent`. Verify delegation.

1. Wrap a remote A2A server (e.g. a Currency Agent) with `RemoteA2aAgent`, pointing at its Agent Card URL.
2. Add it to a parent ADK `Agent` via `sub_agents`.
3. The LLM sees it as a normal sub-agent and delegates with `transfer_to_agent`.

**Expected behaviour:** Remote agent responds via A2A.

In [ ]:
from google.adk.agents import Agent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

# Connect to a remote A2A agent (e.g., Currency Agent)
# Uncomment when a remote agent is running:
# currency_agent = RemoteA2aAgent(
#     name='currency_agent',
#     description='Converts currencies using live exchange rates',
#     agent_card='http://localhost:10000/.well-known/agent.json',
# )
#
# root_with_remote = Agent(
#     model='gemini-3.6-flash',
#     name='documind_connected',
#     instruction='Delegate currency questions to currency_agent.',
#     sub_agents=[currency_agent],
#     tools=[search_documents, summarize_document],
# )
print('RemoteA2aAgent wraps any A2A server as an ADK sub-agent')
print('The LLM sees it as a regular sub-agent (transfer_to_agent)')

## Exercise 5: Currency Agent

**Difficulty:** Medium

Build an MCP-style tool + ADK agent + A2A server — three protocols in one stack.

1. Define a currency-conversion tool (the MCP/tool layer — vertical connection to data).
2. Wrap it in an ADK `LlmAgent` (the reasoning layer).
3. Expose the agent over A2A with `to_a2a` (the horizontal connection to other agents).

**Expected behaviour:** Currency conversion via the MCP+ADK+A2A stack.

In [ ]:
# 1) Tool layer (MCP-style vertical connection to data/rates).
#    In production this would be an MCP server; here it is an ADK function tool.
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount between currencies using a static demo rate table.
    Args:
        amount: Amount to convert.
        from_currency: Source currency code, e.g. USD.
        to_currency: Target currency code, e.g. INR.
    """
    rates = {('USD', 'INR'): USD_INR, ('INR', 'USD'): 1 / USD_INR}
    rate = rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return {'error': f'No rate for {from_currency}->{to_currency}'}
    return {'amount': amount, 'from': from_currency.upper(),
            'to': to_currency.upper(), 'rate': rate,
            'converted': round(amount * rate, 2)}

# 2) Reasoning layer (ADK agent).
currency_agent = LlmAgent(
    name='currency_agent',
    model='gemini-3.6-flash',
    description='Converts currencies using live exchange rates',
    instruction='You convert currencies. Always call convert_currency for math.',
    tools=[convert_currency],
)

# 3) Protocol layer (A2A horizontal connection to other agents).
currency_a2a_app = to_a2a(currency_agent, port=10000)
print('Currency stack ready: tool (MCP-style) + ADK agent + A2A server')
print('Serve: uvicorn currency_agent:currency_a2a_app --port 10000')
print('Sanity check tool:', convert_currency(100, 'USD', 'INR'))

## Exercise 6: Task Lifecycle

**Difficulty:** Medium

Track state transitions: submitted, working, completed.

1. Enumerate the A2A task-lifecycle states, grouped by category (in-progress, interrupted, terminal).
2. Walk a task through a normal happy path: submitted -> working -> completed.
3. Print the progression so the state machine is explicit.

**Expected behaviour:** State progression verified.

In [ ]:
# Task lifecycle states (A2A spec)
states = {
    'in_progress': ['submitted', 'working'],
    'interrupted': ['input-required', 'auth-required'],
    'terminal': ['completed', 'failed', 'canceled', 'rejected']
}
print('Task lifecycle states:')
for category, s in states.items():
    print(f'  {category}: {s}')

# Happy-path progression for a single task
happy_path = ['submitted', 'working', 'completed']
print('\nHappy-path transitions:')
prev = None
for state in happy_path:
    arrow = f'{prev} -> {state}' if prev else f'(start) {state}'
    print(f'  {arrow}')
    prev = state

assert happy_path[-1] in states['terminal'], 'Task must end in a terminal state'
print('\nVerified: task reached a terminal state.')

## Exercise 7: Multi-Vendor Ecosystem

**Difficulty:** Challenge

DocuMind as server + client. Connect to 2 external agents.

1. Keep DocuMind exposed as an A2A server (from Exercise 2).
2. As a client, wrap two external A2A agents — a currency agent and a translation agent — with `RemoteA2aAgent`.
3. Compose them as sub-agents of DocuMind so it can delegate across vendors.

**Expected behaviour:** Cross-agent workflow across vendors.

In [ ]:
from google.adk.agents import Agent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

# DocuMind is already a SERVER via a2a_app (Exercise 2).
# Now make it a CLIENT of two external vendor agents.
# Uncomment when the remote agents are actually running:
# currency_remote = RemoteA2aAgent(
#     name='currency_agent',
#     description='Converts currencies using live exchange rates',
#     agent_card='http://localhost:10000/.well-known/agent.json',
# )
# translation_remote = RemoteA2aAgent(
#     name='translation_agent',
#     description='Translates text between languages',
#     agent_card='http://localhost:10001/.well-known/agent.json',
# )
#
# documind_hub = Agent(
#     model='gemini-3.6-flash',
#     name='documind_hub',
#     instruction=(
#         'You are DocuMind. Summarize documents yourself. '
#         'Delegate currency questions to currency_agent and '
#         'translation requests to translation_agent.'
#     ),
#     tools=[search_documents, summarize_document],
#     sub_agents=[currency_remote, translation_remote],
# )
print('DocuMind = A2A server (a2a_app) + A2A client (2 remote sub-agents)')
print('One agent, both roles: this is the multi-vendor ecosystem pattern.')

## Exercise 8: Deploy to Cloud Run

**Difficulty:** Challenge

Deploy the A2A server to Cloud Run. Test the Agent Card remotely.

1. Package the A2A app (ASGI app served by uvicorn) with a container/source deploy.
2. Deploy to Cloud Run in the target region.
3. Fetch the Agent Card from the public Cloud Run URL to confirm remote discovery.

**Expected behaviour:** Remote Agent Card discoverable.

**Deploy step — run in a terminal / Cloud Shell** from a directory containing your A2A app (`a2a_app` module + `requirements.txt`):

```bash
# Deploy the A2A server from source; the app exposes `a2a_app` served by uvicorn on $PORT.
gcloud run deploy documind-a2a \
  --source . \
  --region us-central1 \
  --allow-unauthenticated \
  --set-env-vars GOOGLE_CLOUD_PROJECT=documind-ai-YOUR-ID,GOOGLE_CLOUD_LOCATION=global,GOOGLE_GENAI_USE_VERTEXAI=TRUE
```

**After the deploy succeeds**, fetch the Agent Card remotely to prove discovery (terminal / Cloud Shell):

```bash
SERVICE_URL=$(gcloud run services describe documind-a2a \
  --region us-central1 --format 'value(status.url)')
echo "Service: $SERVICE_URL"
curl -s "$SERVICE_URL/.well-known/agent.json" | python3 -m json.tool
```